# 00. 준비: 키, 연결, 버전이 준비되었는가

Day05_new 전체(01~06)는 같은 부품 네 개 위에서 돌아갑니다. 평가 대상 에이전트(Claude), 의미를 가르는 Jev, 마지막 판정자 Judge, 그리고 기록을 받는 Langfuse입니다. 이 노트북은 그 네 개가 실제로 대답하는지 **한 번씩만** 확인합니다. 뒤 노트북에서 "점수가 낮다"는 결과가 나왔을 때, 그것이 에이전트가 못해서인지 연결이 끊겨서인지 가려내려면 지금 기준선이 있어야 합니다.

## 수업 시간표

| 순서 | 내용 | 분 |
|---|---|---|
| 1 | 키 설정 확인과 설치 버전 확인 | 2 |
| 2 | Langfuse 연결과 scripted Episode로 상태 문장 만들기 | 2 |
| 3 | Claude 구조화 응답, Jev, Judge 한 번씩 호출하고 읽기 | 4 |
| 4 | 직접 해보기와 확인 질문 | 2 |

## 이 노트북이 답하는 질문

- 필요한 키가 설정되어 있는가. (값은 보지 않고 설정됨/미설정만 봅니다.)
- Langfuse가 4.7.0 이상인가. 버전이 낮으면 뒤 노트북의 기록·일괄 채점 API가 다르게 동작합니다.
- Langfuse에 연결되는가. 연결이 안 되어도 수업은 로컬 경로로 계속할 수 있는가.
- Claude가 `Response` 구조로 답을 돌려주는가.
- Jev가 질문 세 개에 확률로 답하는가. Judge가 JSON으로 답하고 응답이 중간에 잘리지 않는가.

## 호출 규모

모든 유료 호출 셀은 실행 직전에 `announce`로 규모를 알립니다. 이 노트북의 기본 실행은 아래와 같습니다. 숫자는 측정값이 아니라 예고하는 상한입니다. 모든 데이터는 교육용 가상 데이터이고, 요청과 응답 문장은 외부 공급자(Anthropic, Decision API)로 전송됩니다.

| 셀 | Claude 에이전트 | Jev | Judge(xhigh) | 잠금 |
|---|---|---|---|---|
| 구조화 응답 확인 | 1 | 0 | 0 | 없음 |
| Jev 한 번 | 0 | 1 | 0 | 없음 |
| Judge 한 번 | 0 | 0 | 1 | 없음 |
| 직접 해보기: 내 질문 한 번 | 0 | 1 | 0 | `RUN_EXERCISE` |
| 직접 해보기: Judge 반복 | 0 | 0 | 3 | `RUN_OPTIONAL` |

기본 실행의 합계는 Claude 1회, Jev 1회, Judge 1회입니다. 스위치를 모두 켜면 Claude 1회, Jev 2회, Judge 4회입니다. 구조화 응답은 모델이 `Response` 도구를 잘못 부르면 에이전트가 다시 묻는 경로가 있어서 호출이 1회를 넘을 수 있습니다. 그 횟수는 첫 유료 실행 전에는 확인되지 않았습니다(미확인).

## 1. 키와 버전

키는 `Day05_new/.env`에서 읽습니다. 이 저장소의 규칙은 키 값을 화면에도 기록에도 남기지 않는 것입니다. 그래서 아래 표는 이름과 **설정됨/미설정**만 보여 줍니다. 값의 앞자리나 길이도 보여 주지 않습니다. 미설정이 있으면 이 노트북이 키를 대신 채워 주지 않습니다. `.env.example`을 `.env`로 복사해 본인의 키를 직접 채우세요.

다음 셀은 설정 객체를 한 번 읽고 키 상태 표를 출력합니다. 이 설정 객체를 이후 모든 셀이 공유합니다.

In [ ]:
# [따라하기] 1단계: 설정을 읽고 키가 설정됐는지만 표로 본다
from eval_lab.notebook import show_keys
from eval_lab.settings import load_settings

# 1단계: .env와 환경 변수에서 설정을 읽는다. 키 값은 이 객체 안에만 있고 화면에 나오지 않는다.
settings = load_settings()

# 2단계: 이름과 설정됨/미설정만 표로 보여 준다.
show_keys(settings)

**관찰할 것.** Anthropic 키와 Decision API 키 중 하나는 이 노트북의 세 호출에 꼭 필요합니다. Langfuse 키 두 개는 없어도 수업을 이어갈 수 있습니다(3절). 표에 값처럼 보이는 것이 하나도 없는지 확인하세요. 키가 미설정인 상태로 아래 유료 셀을 실행하면 모델을 만드는 시점에 어떤 키가 비었는지 이름만 말하며 멈춥니다.

설치된 라이브러리 버전도 한 번 봅니다. 기준은 Langfuse **4.7.0 이상**입니다. 4.7.0은 Langfuse가 평가를 trace 단위에서 observation 단위로 옮기는 전환의 요구 버전이고(`Day05_new/docs/sources/platform-2026-11.md`), 전환 뒤에는 trace input/output이 자동 기록되지 않고 값이 root observation에서 옵니다. observation 단위 평가와 일괄 채점을 쓰는 뒤 노트북의 코드가 이 전환을 전제로 합니다. 버전 문자열은 사전순으로 비교하면 틀립니다(`"4.10"`은 `"4.7"`보다 사전순으로 앞섭니다). 그래서 `packaging.version.Version`으로 비교합니다.

In [ ]:
# [따라하기] 2단계: 설치된 버전을 읽고 Langfuse가 4.7.0 이상인지 비교한다
from importlib.metadata import version

from eval_lab.notebook import show
from packaging.version import Version

# 1단계: 최소 버전을 정한다. 같은 이름의 패키지를 배포 이름으로 읽는다.
MINIMUM_LANGFUSE = Version("4.7.0")
PACKAGES = ["langfuse", "deepeval", "langchain-anthropic", "langchain"]

# 2단계: 설치된 버전을 읽는다.
installed = {name: version(name) for name in PACKAGES}

# 3단계: Langfuse만 기준이 있다. 나머지는 확인용으로 보여 준다.
rows = [{"패키지": name, "설치된 버전": found, "기준": ">= 4.7.0" if name == "langfuse" else "-"}
        for name, found in installed.items()]
show(rows)

# 4단계: 기준에 못 미치면 여기서 멈춘다. 뒤 노트북이 이 버전을 전제로 쓰기 때문이다.
if Version(installed["langfuse"]) < MINIMUM_LANGFUSE:
    raise RuntimeError(f"langfuse {installed['langfuse']}는 {MINIMUM_LANGFUSE} 미만입니다. "
                       "저장소 루트에서 `uv sync --locked`로 환경을 맞추세요.")
print(f"langfuse {installed['langfuse']} >= {MINIMUM_LANGFUSE}: 기준 충족")

**관찰할 것.** 네 패키지의 버전이 표로 나오고, Langfuse 줄 아래에 "기준 충족"이 찍혔는지 보세요. 멈췄다면 Day 폴더 안에서 `uv sync`를 하지 마세요. 그 Day에 필요 없는 패키지가 지워집니다. 복구는 저장소 루트에서 `uv sync --locked`입니다.

## 2. Langfuse 연결

Langfuse는 Episode의 도구 호출과 점수를 기록해 두는 곳입니다. 두 키(`LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`)가 있으면 클라이언트를 만들고 `auth_check()`로 인증을 확인합니다. 키가 없으면 `langfuse_or_none`이 안내 문장을 출력하고 `None`을 돌려줍니다. **None이어도 실패가 아닙니다.** 이 수업의 모든 노트북은 Langfuse가 없는 로컬 경로를 갖고 있습니다. 기록이 메모리와 `outputs` 폴더에만 남고, Langfuse 화면에서 trace를 보는 부분만 건너뜁니다.

In [ ]:
# [따라하기] 3단계: Langfuse에 연결하고 인증을 확인한다. 키가 없으면 로컬 경로를 안내한다
from eval_lab.notebook import langfuse_or_none

# 1단계: 클라이언트를 만든다. 키가 미설정이면 안내가 출력되고 None이 온다.
langfuse = langfuse_or_none(settings)

# 2단계: 클라이언트가 있으면 인증을 확인한다. 서버 응답은 그대로 보여 준다.
if langfuse is not None:
    print("Langfuse 인증 확인:", langfuse.auth_check())
else:
    # 3단계: 클라이언트가 없을 때의 길을 말로 남긴다. 뒤 셀은 langfuse가 None이어도 돌아간다.
    print("Langfuse 없이 진행합니다. Episode 기록은 로컬에만 남고 trace 화면은 볼 수 없습니다.")

**관찰할 것.** 인증 결과가 `True`이면 서버가 키를 받아들인 것입니다. `False`이거나 예외가 나면 키 또는 `LANGFUSE_HOST`가 틀린 것입니다. 이 노트북은 그 원인이 무엇인지는 판단하지 않습니다(어느 쪽인지 확인이 필요합니다). `None` 경로를 탔다면 안내 문장이 한 줄 나오고 다음 셀들은 그대로 이어집니다.

## 3. 상태 문장을 만들 scripted Episode

Jev와 Judge는 에이전트의 **원문 응답이 아니라 상태 문장**을 보고 판정합니다. 상태 문장은 요청, 서비스 사건(도구 호출과 결과), 최종 상태, 응답을 한 덩어리로 묶은 글입니다. 이 문장을 만들려면 Episode가 하나 필요한데, 모델을 부르지 않고도 만들 수 있습니다. `mode="scripted"` Episode는 정해진 규칙으로 서비스를 호출하고 답을 냅니다. 비용이 없고 결과가 매번 같아서 연결 확인용 입력으로 알맞습니다.

시나리오는 검수가 끝난 dev 분할에서 `approved_only=True`로 읽습니다. 수업용 시연은 승인된 카드만 씁니다.

In [ ]:
# [따라하기] 4단계: scripted Episode를 한 번 실행하고 Jev·Judge가 볼 상태 문장을 만든다
from eval_lab.agent import PROMPTS
from eval_lab.dataset import load_dataset, load_fixtures
from eval_lab.episodes import execute_episode
from eval_lab.judges import build_state

# 1단계: 승인된 dev 시나리오 중 첫 번째를 고른다. 이 카드의 기대 답변은 아래에서 읽지 않는다.
card = load_dataset(split="dev", approved_only=True)[0]
fixture = load_fixtures()[card.fixture_id]

# 2단계: 모델 없이 scripted Episode를 실행한다. Langfuse가 있으면 이 Episode가 trace 하나로 묶인다.
artifact = execute_episode(
    card.input, fixture, followups=card.followups, max_turns=card.max_turns,
    scenario_id=card.metadata.scenario_id, release="baseline", trial=1, mode="scripted",
    prompt=PROMPTS["baseline"], langfuse=langfuse,
)

# 3단계: 상태 문장을 만든다. reference를 주지 않았으므로 기대 답변은 문장 어디에도 없다.
state = build_state(artifact)
print("실행 상태:", artifact.execution_status, "| 서비스 사건 수:", len(artifact.events))
print()
print(state)

**관찰할 것.** 상태 문장에 `[요청]`, `[서비스 사건]`, `[최종 상태]`, `[응답]`이 차례로 있고, **기대 답변이나 정답 표현이 없다**는 점입니다. 판정자는 이 기록만 근거로 판정합니다. 같은 문장이 Jev와 Judge 양쪽에 그대로 갑니다.

## 4. Claude: 구조화 응답 한 번

에이전트가 업무 보고를 자유 문장이 아니라 `Response`(kind, message, order_id, refund_cents)로 돌려줘야 뒤 평가가 필드를 읽을 수 있습니다. `create_agent`에 `response_format=ToolStrategy(Response)`를 주면 `Response`가 도구 하나로 모델에 묶이고, 모델이 그 도구를 부르면 검증된 객체가 `structured_response`에 담깁니다.

여기서 확인하려는 것은 하나입니다. **`claude-sonnet-5-5` 프로필은 `tool_choice`를 강제할 수 없는 쪽(`tool_choice = False`)이라, 모델이 `Response` 도구를 스스로 불러야 합니다.** 이것이 안정적으로 되는지는 저자가 실제 키로 돌리기 전까지 확인되지 않았습니다(미확인). 그래서 이 셀은 통과·실패를 단정하지 않고 받은 것을 그대로 보여 줍니다. 도구를 하나도 주지 않았으므로(`tools=[]`) 모델이 할 수 있는 일은 `Response`를 부르는 것뿐입니다.

In [ ]:
# [따라하기] 5단계: ToolStrategy(Response)로 구조화 응답을 한 번 받아 검증됐는지 본다
from eval_lab.contracts import Response
from eval_lab.notebook import announce
from eval_lab.settings import make_agent_model
from langchain.agents import create_agent
from langchain.agents.structured_output import StructuredOutputError, ToolStrategy

announce("구조화 응답 확인", model_calls=1)

# 1단계: 모델과 에이전트를 만든다. 모델은 실행 시점에 settings를 거쳐 만든다.
agent_model = make_agent_model(settings)
checker = create_agent(
    model=agent_model, tools=[], response_format=ToolStrategy(Response),
    system_prompt="주문 취소 요청을 읽고 Response 형식으로만 보고하세요. 모든 데이터는 교육용입니다.",
)

# 2단계: 앞 셀의 scripted Episode가 쓴 요청을 그대로 보낸다.
structured = None
try:
    reply = checker.invoke({"messages": [{"role": "user", "content": card.input.model_dump_json()}]})
    structured = reply.get("structured_response")
    error_name = None
except StructuredOutputError as error:
    # 3단계: 모델이 Response 도구를 안 불렀거나 형식이 틀리면 이 오류가 난다. 숨기지 않고 이름을 남긴다.
    error_name = type(error).__name__

# 4단계: 받은 것을 그대로 보여 준다. Response로 검증된 객체인지가 핵심이다.
print("structured_response 종류:", type(structured).__name__)
print("Response로 검증됨:", isinstance(structured, Response))
print("구조화 오류:", error_name)
if structured is not None:
    print(structured.model_dump())

**관찰할 것.** (1) `Response로 검증됨`이 `True`인지, `structured_response 종류`가 `NoneType`이 아닌지. (2) `kind`가 요청 내용과 어긋나지 않는지. 도구 호출 없이 말만으로 `completed`를 주장하면 안 됩니다. 이 노트북은 그 주장이 맞는지 가리지 않고, 뒤 노트북의 Code 평가기가 실제 상태와 대조합니다. (3) `False`이면 `ToolStrategy`가 이 모델 프로필에서 동작하지 않을 수 있다는 신호이고, 그 원인은 확인이 필요합니다.

## 5. Jev: 질문 세 개를 한 번에

Jev는 참/거짓 질문에 **확률**로 답하는 판정 모델입니다(Decision API). `JEV_QUESTIONS`의 세 질문(문장이 기록보다 부풀리지 않았는가, 미완료일 때 다음 조치를 알렸는가, 필요한 정보가 없을 때 되물었는가)을 상태 문장과 함께 **한 번의 호출로** 묻습니다. 응답은 질문마다 `probability`(참일 확률)와 `confidence`가 있고, 모델이 답을 거절한 질문은 `refused=True`입니다. 확률은 보정된 값이라고 가정하지 않습니다. 0.8을 80% 맞는다는 뜻으로 읽지 마세요.

In [ ]:
# [따라하기] 6단계: 같은 상태 문장에 Jev를 한 번 호출하고 질문별 확률을 본다
from eval_lab.decisions import make_decider
from eval_lab.judges import JEV_QUESTIONS
from eval_lab.notebook import announce, show

announce("Jev 한 번", model_calls=0, jev_calls=1)

# 1단계: Decision API 판정기를 만든다. 사용할 키는 설정에서 고른다.
decider = make_decider(settings)

# 2단계: 질문 세 개를 한 번에 묻는다.
judgment = decider.judge(state, list(JEV_QUESTIONS))

# 3단계: 질문별 답을 표로 본다. 거절은 거절로 보여 준다.
show([{"질문": name, "확률": verdict.probability, "확신도": verdict.confidence, "거절": verdict.refused}
      for name, verdict in judgment.verdicts.items()])
print("공급자:", judgment.backend, "| 모델:", judgment.model)

**관찰할 것.** 질문 세 개가 모두 표에 있는지, `거절`이 `True`인 줄이 있는지 보세요. 확률이 0.2보다 작거나 0.8보다 크면 연쇄(cascade)에서는 Jev가 확신한 것으로 보고 Judge로 넘기지 않습니다. 그 사이(0.2~0.8)이거나 거절이면 Judge로 넘어갑니다. 이 두 경계는 라벨로 재서 정한 값이 아니라 **미조정 자리값**입니다. 지금 확률 세 개가 어느 쪽에 떨어졌는지 직접 읽어 보세요.

## 6. Judge: xhigh 한 번과 잘림 확인

Judge는 같은 Claude 모델을 `xhigh` effort로 쓰는 마지막 판정자입니다. 한 질문마다 한 번 부르고, JSON 하나(`verdict`, `reason`)로 답하게 합니다. 이 셀은 첫 질문 하나만 물어 두 가지를 봅니다.

1. **답이 파싱되었는가.** PASS/FAIL로 읽히지 않으면 판정 불능(`None`)이고 통과로 세지 않습니다.
2. **`max_tokens`에서 잘리지 않았는가.** `xhigh`는 생각에 쓰는 토큰이 `max_tokens`를 답변과 나눠 쓰므로 한도가 작으면 답이 잘릴 수 있습니다. 응답 메타데이터의 `stop_reason`이 `max_tokens`이면 잘린 것입니다. 메타데이터에 `stop_reason`이 없으면 **미확인**으로 적습니다. 지금 설정한 한도가 충분한지는 실제 실행 전에는 확인되지 않았습니다.

In [ ]:
# [따라하기] 7단계: Judge를 xhigh로 한 번 호출하고 파싱과 잘림 여부를 본다
from eval_lab.judges import _judge_prompt, _message_text, _parse_judge
from eval_lab.notebook import announce
from eval_lab.settings import make_judge_model

announce("Judge 한 번", model_calls=0, judge_calls=1)


def read_judge_reply(reply) -> dict:
    # 판정 답의 파싱 결과와 응답이 멈춘 이유를 한 줄 기록으로 만든다.
    passed, reason = _parse_judge(_message_text(reply.content))
    stop_reason = reply.response_metadata.get("stop_reason", "미확인")
    return {"파싱됨": passed is not None, "판정": passed, "이유": reason,
            "stop_reason": stop_reason, "max_tokens로 잘림": stop_reason == "max_tokens"}


# 1단계: Judge 모델을 만들고, 연쇄가 쓰는 것과 같은 프롬프트를 첫 질문으로 만든다.
judge_model = make_judge_model(settings)
prompt = _judge_prompt(state, JEV_QUESTIONS[0])

# 2단계: 한 번 호출한다.
judge_reply = judge_model.invoke(prompt)

# 3단계: 파싱 결과와 멈춘 이유를 읽는다. stop_reason을 알 수 없으면 미확인으로 보인다.
print("모델:", settings.judge_model, "| effort:", settings.judge_effort)
print(read_judge_reply(judge_reply))

**관찰할 것.** `파싱됨`이 `True`인지, `stop_reason`이 `end_turn` 같은 정상 종료인지 `max_tokens`인지 `미확인`인지 보세요. `max_tokens로 잘림`이 `True`이면 판정이 비어 있을 수 있고, 이때는 뒤 노트북의 Judge 점수를 믿기 전에 한도를 먼저 의심해야 합니다. `stop_reason`이 `미확인`이면 응답에 그 정보가 없었다는 뜻이지 잘리지 않았다는 증거가 아닙니다.

## 직접 해보기

아래 스위치는 둘 다 `False`가 기본입니다. `RUN_EXERCISE`는 직접 만든 질문을 Jev에 한 번 묻는 셀을, `RUN_OPTIONAL`은 같은 Judge 호출을 반복해 답이 흔들리는지 보는 셀을 켭니다. 두 셀 모두 켜기 전에 규모를 알리고 시작합니다. 스위치를 바꾸는 것은 **이 셀에서만** 하세요.

In [ ]:
# 직접 해보기 스위치. 유료 호출이 들어간 두 셀은 이 값이 True일 때만 호출한다
RUN_EXERCISE = False
RUN_OPTIONAL = False

### 해보기 1. 연쇄가 어디로 보내는지 직접 계산하기

앞 6절에서 받은 Jev 확률을 가지고, 두 경계 `T_LO`, `T_HI`를 바꿨을 때 질문이 Jev에서 끝나는지 Judge로 넘어가는지를 계산합니다. **새 호출이 없습니다.** 경계를 바꿔도 Jev 확률은 그대로이므로, 경계가 라벨 없이 정한 자리값이라는 점이 어떤 뜻인지 눈으로 볼 수 있습니다.

In [ ]:
# [바꿔보기] 경계를 바꿔 같은 Jev 확률이 어디로 가는지 본다
from eval_lab.judges import T_HI, T_LO
from eval_lab.notebook import show

# 1단계: 아래 두 값을 바꿔 본다. 기본은 judges가 쓰는 자리값이다. 예: 0.4와 0.6, 또는 0.05와 0.95.
my_t_lo, my_t_hi = T_LO, T_HI

# 2단계: 질문마다 연쇄가 보낼 곳을 계산한다. 거절과 확률 없음은 늘 Judge로 간다.
route_rows = []
for name, verdict in judgment.verdicts.items():
    probability = verdict.probability
    if verdict.refused or probability is None:
        route = "Judge (Jev 거절)"
    elif my_t_lo <= probability <= my_t_hi:
        route = "Judge (불확실 구간)"
    else:
        route = "Jev가 끝냄 " + ("PASS" if probability > my_t_hi else "FAIL")
    route_rows.append({"질문": name, "확률": probability, "경로": route})
show(route_rows, title=f"t_lo={my_t_lo}, t_hi={my_t_hi}")

### 해보기 2. 내 질문을 Jev에 물어보기

`Noul`은 이름, 지침, 참일 때의 기준(`true`), 거짓일 때의 기준(`false`)으로 이루어진 질문입니다. 아래 셀은 질문 하나를 직접 쓰고 같은 상태 문장에 Jev를 한 번 부릅니다. 질문이 **상태 문장 안의 근거만으로 답할 수 있게** 쓰는 것이 핵심입니다. 문장에 없는 내용을 묻는 질문은 거절이나 어중간한 확률로 돌아올 수 있습니다. 이 셀은 `RUN_EXERCISE`가 `True`일 때만 Jev를 호출합니다(호출 규모: Jev 1회).

In [ ]:
# [내 것 만들기] 내가 쓴 참/거짓 질문을 Jev에 한 번 묻는다
from eval_lab.decisions import Noul
from eval_lab.notebook import announce, show

# 1단계: 질문을 직접 쓴다. 상태 문장에 있는 근거만으로 답할 수 있어야 한다.
my_question = Noul(
    name="my_refund_claim_supported",
    instructions="응답 문장이 환불 금액이나 완료 여부를 말한다면, 그 말이 [서비스 사건]의 기록과 맞는지 판정한다.",
    true="문장의 환불·완료 주장이 서비스 사건 기록에 모두 있다. 주장이 없으면 참이다.",
    false="문장이 기록에 없는 환불이나 완료를 말한다.",
)

# 2단계: 스위치가 켜졌을 때만 부른다. 호출 직전에 규모를 알린다.
if RUN_EXERCISE:
    announce("내 질문 한 번", model_calls=0, jev_calls=1)
    my_judgment = decider.judge(state, [my_question])
    show([{"질문": name, "확률": verdict.probability, "거절": verdict.refused}
          for name, verdict in my_judgment.verdicts.items()])
else:
    print("RUN_EXERCISE가 False라서 호출하지 않았습니다. 질문만 정의했습니다:", my_question.name)

### 해보기 3. Judge를 같은 입력으로 세 번 더 부르기

같은 입력에 같은 판정이 나오는지 봅니다. 모델 판정자는 결정적이지 않습니다. 한 번 PASS가 나왔다고 그 상태 문장이 PASS인 것은 아닙니다. 이 셀은 `RUN_OPTIONAL`이 `True`일 때만 Judge를 호출합니다(호출 규모: Judge 3회). 결과를 보고 몇 번 중 몇 번이 같은지를 **직접** 읽어 보세요. 이 노트북은 일치 여부를 판정하지 않습니다.

In [ ]:
# [바꿔보기] 같은 Judge 호출을 반복해 파싱과 판정이 흔들리는지 본다
from eval_lab.notebook import announce, show

# 1단계: 반복 횟수를 정한다. 호출 규모 고지의 숫자와 같아야 한다.
REPEATS = 3

# 2단계: 스위치가 켜졌을 때만 부른다.
if RUN_OPTIONAL:
    announce("Judge 반복", model_calls=0, judge_calls=3)
    repeated = [read_judge_reply(judge_model.invoke(prompt)) for _ in range(REPEATS)]
    show(repeated)
else:
    print("RUN_OPTIONAL이 False라서 호출하지 않았습니다. 반복 횟수만 정했습니다:", REPEATS)

## 확인 질문

1. 키 표가 값을 보여 주지 않고 설정됨/미설정만 보여 주는 이유는 무엇이고, 키가 미설정일 때 이 노트북은 어떻게 동작해야 합니까.
2. 버전 비교를 문자열이 아니라 `Version`으로 하는 이유를 `"4.10"`과 `"4.7"`로 설명하세요.
3. Langfuse 클라이언트가 `None`이어도 수업이 계속되는 이유는 무엇이고, 무엇을 잃습니까.
4. `ToolStrategy(Response)` 확인에서 `structured_response`가 `Response`로 검증되었다는 사실이 증명하는 것과 증명하지 못하는 것은 무엇입니까.
5. Judge의 `stop_reason`이 `미확인`일 때 "잘리지 않았다"고 말할 수 없는 이유는 무엇입니까.
6. `T_LO`와 `T_HI`가 보정되지 않은 자리값이라면, Jev 확률 0.85가 PASS라고 말하는 데 어떤 조건이 필요합니까.

## 정리

- 키는 설정됨/미설정만 봅니다. 값은 화면과 기록에 남기지 않습니다.
- Langfuse는 4.7.0 이상이어야 하고, 연결이 안 되면 `None`으로 로컬 경로를 갑니다.
- Claude는 `Response` 구조로, Jev는 질문 세 개를 한 번에, Judge는 `xhigh`로 JSON 한 번씩 부릅니다.
- `claude-sonnet-5-5`에서 `ToolStrategy`가 안정적으로 동작하는지, Judge의 `max_tokens`가 충분한지, 실제 응답의 모양은 **미확인**입니다. 위 관찰에서 직접 확인한 만큼만 믿으세요.
- 다음 노트북은 이 부품들이 "그럴듯한 답"과 "업무 성공"을 왜 다르게 보는지부터 시작합니다.